# AlertManager routing — HOBO-281 contract

`evaluateAlertRouting` is a pure, event-sourced routing evaluation: related SLO burns **group** into one receiver delivery (`groupKey=sloName=...`), an active **silence** suppresses paging, a critical root cause **inhibits** dependent warnings in the same tenant, and a failing receiver produces a **durable retry** (`pendingDeliveries` + `alert.Routed status=retry-scheduled`). Every decision is replayable from the emitted events via `replayAlertEvents`, which explains per-fingerprint why something did or did not page.

In [1]:
import { strict as assert } from "node:assert";

import {
	ALERT_ROUTING_CONTRACT_VERSION,
	evaluateAlertRouting,
	replayAlertEvents,
} from "./src/alertmanager-routing.ts";

const NOW = "2026-06-26T06:45:00.000Z";

const basePolicy = () => ({
	groupBy: ["sloName"],
	dedupWindow: "5m",
	routes: [
		{ id: "slo-pages", matcher: { kind: "slo" }, receiverId: "pagerduty" },
		{ id: "health-pages", matcher: { kind: "health" }, receiverId: "pagerduty" },
	],
});

const sloBurn = (id: string, overrides: Record<string, unknown> = {}) => ({
	id,
	kind: "slo",
	type: "slo.BudgetBurned",
	status: "firing",
	startsAt: NOW,
	labels: {
		sloName: "checkout-api",
		severity: "page",
		window: (overrides.window as string) ?? "fast-burn",
	},
	metric: { burnRate: (overrides.burnRate as number) ?? 10, remainingBudgetRatio: 0.1 },
});

const recordingReceiver = (receiverId: string) => ({
	send: (delivery: { groupKey: string }) => ({
		ok: true,
		receiverId,
		providerMessageId: `${receiverId}:${delivery.groupKey}`,
	}),
});

// 1) Two related SLO burns group into ONE receiver notification.
const grouped = evaluateAlertRouting({
	now: NOW,
	signals: [
		sloBurn("checkout-fast", { burnRate: 16.2 }),
		sloBurn("checkout-slow", { burnRate: 2.5, window: "slow-burn" }),
	],
	policy: basePolicy(),
	receivers: { pagerduty: recordingReceiver("pagerduty") },
});
assert.equal(grouped.schemaVersion, ALERT_ROUTING_CONTRACT_VERSION);
assert.equal(grouped.deliveries.length, 1);
assert.equal(grouped.deliveries[0].alertFingerprints.length, 2);
const groupEvent = grouped.events.find((event: { type: string }) => event.type === "alert.Grouped");
assert.equal(groupEvent.data.groupKey, "sloName=checkout-api");
console.log(`schema: ${grouped.schemaVersion}`);
console.log(
	`grouping: deliveries=${grouped.deliveries.length} alerts=${grouped.deliveries[0].alertFingerprints.length} groupKey=${groupEvent.data.groupKey}`,
);
console.log(`events: ${grouped.events.map((event: { type: string }) => event.type).join(" -> ")}`);

// 2) An active silence suppresses paging; replay explains WHY no page fired.
const silenced = evaluateAlertRouting({
	now: NOW,
	signals: [sloBurn("checkout-fast", { burnRate: 16.2 })],
	policy: {
		...basePolicy(),
		silences: [
			{
				id: "silence-maintenance",
				matcher: { sloName: "checkout-api" },
				startsAt: "2026-06-26T06:00:00.000Z",
				endsAt: "2026-06-26T07:00:00.000Z",
			},
		],
	},
	receivers: { pagerduty: recordingReceiver("pagerduty") },
});
assert.equal(silenced.deliveries.length, 0);
const silencedExplanation = replayAlertEvents(silenced.events).explanations[
	silenced.events[0].data.fingerprint
];
assert.equal(silencedExplanation.status, "silenced");
assert.equal(silencedExplanation.reason, "silence-maintenance");
console.log(
	`silence: deliveries=${silenced.deliveries.length} replay status=${silencedExplanation.status} reason=${silencedExplanation.reason}`,
);

// 3) Inhibition: a critical root cause mutes the dependent warning in the same tenant.
const inhibited = evaluateAlertRouting({
	now: NOW,
	signals: [
		{
			id: "gateway-down",
			kind: "health",
			type: "health.Degraded",
			status: "firing",
			startsAt: NOW,
			labels: { service: "gateway", severity: "critical", tenant: "tenant-a" },
			metric: {},
		},
		{
			id: "checkout-burn",
			kind: "slo",
			type: "health.Degraded",
			status: "firing",
			startsAt: NOW,
			labels: { service: "checkout-api", severity: "warning", tenant: "tenant-a" },
			metric: {},
		},
	],
	policy: {
		...basePolicy(),
		groupBy: ["tenant"],
		inhibitions: [
			{
				id: "root-cause-critical-suppresses-warning",
				sourceMatcher: { severity: "critical" },
				targetMatcher: { severity: "warning" },
				equal: ["tenant"],
			},
		],
	},
	receivers: { pagerduty: recordingReceiver("pagerduty") },
});
assert.deepEqual(
	inhibited.deliveries.map((delivery: { alertFingerprints: string[] }) => delivery.alertFingerprints.length),
	[1],
);
const inhibitedEntry = Object.values(replayAlertEvents(inhibited.events).explanations).find(
	(entry: { status: string }) => entry.status === "inhibited",
) as { status: string; reason: string };
assert.equal(inhibitedEntry.reason, "root-cause-critical-suppresses-warning");
console.log(
	`inhibition: delivered=${inhibited.deliveries[0].alertFingerprints.length} muted reason=${inhibitedEntry.reason}`,
);

// 4) Receiver failure keeps the alert and schedules a durable retry.
const failed = evaluateAlertRouting({
	now: NOW,
	signals: [sloBurn("checkout-fast", { burnRate: 16.2 })],
	policy: basePolicy(),
	receivers: {
		pagerduty: { send: () => ({ ok: false, receiverId: "pagerduty", retryAfter: "30s", error: "provider-down" }) },
	},
});
assert.equal(failed.deliveries.length, 0);
assert.equal(failed.pendingDeliveries.length, 1);
const routedEvent = failed.events.find((event: { type: string }) => event.type === "alert.Routed");
assert.equal(routedEvent.data.status, "retry-scheduled");
assert.equal(replayAlertEvents(failed.events).pendingDeliveries.length, 1);
console.log(
	`receiver failure: pending=${failed.pendingDeliveries.length} routed status=${routedEvent.data.status}`,
);

console.log("CONTRACT HELD: group, silence, inhibit, and retry — all replayable from events");

schema: hobo.alert-routing/v1
grouping: deliveries=1 alerts=2 groupKey=sloName=checkout-api
events: alert.Fired -> alert.Fired -> alert.Grouped -> alert.Routed
silence: deliveries=0 replay status=silenced reason=silence-maintenance
inhibition: delivered=1 muted reason=root-cause-critical-suppresses-warning
receiver failure: pending=1 routed status=retry-scheduled
CONTRACT HELD: group, silence, inhibit, and retry — all replayable from events

The assertions above all passed and the printed lines match `examples/usage.expected.txt` — the same run the draft-artifact gate re-executes and diffs.